# 🤝 vdclient sdk 使用教學 (操作 RDB)

* 取得資料源連線資訊

| 類別/模組 | 功能 | 備註 |
|-----------|------|------|
| `get_datasource_config(datasource)` | 取得完整連線設定 | 使用 DeepFlow SDK 取得連線設定 |


* 取得資料源連線 connection 
| 類別/模組 | 功能 | 備註 |
|-----------|------|------|
| ✅`rdb_connection(datasource)` | 回傳 SQLAlchemy connection | 支援 PostgreSQL, SQL Server, Oracle |
| `sftp_client(datasource)` | 回傳 Paramiko SFTPClient | 可用於下載、上傳、列出遠端檔案 |
| `mongo_client(datasource)` | 回傳 pymongo.MongoClient | 支援 Mongo 資料庫的操作 |


## 🛢️rdb_connection 範例說明

本教學將展示如何使用 `vdclient_magic.core.connectors.rdb_connection()` 與 SQLAlchemy engine 操作資料庫，包含：

1. 查詢資料
2. 建立資料表
3. 插入資料（單筆、批次、DataFrame）
4. 更新 / 刪除
5. 使用 Pandas 匯入大量資料


In [1]:
import pandas as pd
from sqlalchemy import text
from vdclient_magic.core.connectors import rdb_connection

In [2]:
datasource = 'oracledata'  # 在 DeepFlow 資料源連線管理上設定的「資料源名稱」

### 🔍 查詢資料

In [3]:
with rdb_connection(datasource) as conn:
    limit_cond =  ' LIMIT 10'
    
    if datasource == 'oracledata':
        limit_cond = ' FETCH FIRST 10 ROWS ONLY' #oracle
    
    df = pd.read_sql(text("SELECT * FROM provider_data" + limit_cond), conn)


In [4]:
df.shape

(10, 20)

In [5]:
with rdb_connection(datasource) as conn:
    df = pd.read_sql(text("SELECT * FROM provider_data"), conn)

In [6]:
df.shape

(299717, 20)

## 🏗️ 建立資料表（不影響原始資料）

In [7]:
from vdclient_magic.core.connectors import rdb_connection

with rdb_connection(datasource) as conn:
    with conn.begin():
        conn.execute(text("""
            CREATE TABLE tmp_test_table (
                id INTEGER PRIMARY KEY,
                name VARCHAR(100)
            )
        """))
    print("✅ 建立 tmp_test_table 成功")


✅ 建立 tmp_test_table 成功


## ➕ 插入資料

In [8]:
with rdb_connection(datasource) as conn:
    with conn.begin():
        if datasource =='oracledata': #oracle
            conn.execute(text("""
                INSERT ALL
                    INTO tmp_test_table (id, name) VALUES (1, 'Alice')
                    INTO tmp_test_table (id, name) VALUES (2, 'Bob')
                SELECT 1 FROM dual
            """))
        else:
            conn.execute(text("""
                INSERT INTO tmp_test_table (id, name)
                VALUES (1, 'Alice'), (2, 'Bob')
            """))
    print("✅ 新增資料成功")


✅ 新增資料成功


## ❌ 刪除資料

In [9]:
with rdb_connection(datasource) as conn:
    with conn.begin():
        conn.execute(text("""
            DELETE FROM tmp_test_table WHERE id = 1
        """))
    print("✅ 刪除 ID = 1 的資料成功")


✅ 刪除 ID = 1 的資料成功


## 🧱 修改欄位結構

In [10]:
with rdb_connection(datasource) as conn:
    with conn.begin():
        if datasource =='oracledata': #oracle
            conn.execute(text("""
                ALTER TABLE tmp_test_table ADD email VARCHAR2(100)
            """))
        else:
            conn.execute(text("""
                ALTER TABLE tmp_test_table ADD COLUMN email VARCHAR(100)
            """))
    print("✅ 新增欄位 email 成功")


✅ 新增欄位 email 成功


## 🧨 刪除資料表

In [11]:
with rdb_connection(datasource) as conn:
    with conn.begin():
        conn.execute(text("DROP TABLE tmp_test_table"))
    print("✅ 刪除 tmp_test_table 成功")


✅ 刪除 tmp_test_table 成功


## 📦 DataFrame 批次插入大量資料

In [12]:
# 建立連線與 engine
conn = rdb_connection(datasource)
engine = conn.engine
table_name = 'tmp_massive_insert'

# 2. 建立測試資料表
with engine.begin() as conn:
    # 先檢查是否存在
    if datasource == 'oracledata':
        exists = conn.execute(
            text("""
                SELECT COUNT(*) 
                FROM user_tables 
                WHERE table_name = :table_name
            """),
            {"table_name": table_name.upper()}  # 將表名轉大寫傳入
        ).scalar()

        if exists == 0:
            conn.execute(text(f"""
                CREATE TABLE """ + table_name + """  (
                    id INTEGER PRIMARY KEY,
                    name VARCHAR2(100)
                )
            """))
    else:    
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS """ + table_name + """ (
                id INTEGER PRIMARY KEY,
               name VARCHAR(100)
            )
        """))
print("✅ 建立 " + table_name + " 成功")

# 3. 建立 50 萬筆 DataFrame 資料
df = pd.DataFrame({
    "id": range(1, 500_001),
    "name": ["測試資料"] * 500_000
})
print("✅ 模擬出 500,000 筆資料")

# 4. 分批寫入（含交易控制 rollback on error）
chunksize = 50_000
try:
    with engine.begin() as conn:  # 🔁 若有任何一個 insert 失敗，會 rollback
        for i, start in enumerate(range(0, len(df), chunksize), start=1):
            df_chunk = df.iloc[start:start + chunksize]
            # 將資料轉成 dict list
            data = df_chunk.to_dict(orient="records")

            # 使用 executemany 插入
            conn.execute(
                text("INSERT INTO "+table_name+" (id, name) VALUES (:id, :name)"),
                data
            )
            #df_chunk.to_sql("tmp_massive_insert", con=conn, index=False, if_exists="append", method="multi")
            print(f"✅ 第 {i} 批寫入完成 ({len(df_chunk):,} 筆)")
except Exception as e:
    print(f"❌ 批次寫入失敗，已自動 rollback：{e}")


# 5. 確認筆數
with engine.begin() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM "+table_name)).scalar()
    print(f"📊 資料表總筆數: {total:,} 筆")

✅ 建立 tmp_massive_insert 成功
✅ 模擬出 500,000 筆資料
✅ 第 1 批寫入完成 (50,000 筆)
✅ 第 2 批寫入完成 (50,000 筆)
✅ 第 3 批寫入完成 (50,000 筆)
✅ 第 4 批寫入完成 (50,000 筆)
✅ 第 5 批寫入完成 (50,000 筆)
✅ 第 6 批寫入完成 (50,000 筆)
✅ 第 7 批寫入完成 (50,000 筆)
✅ 第 8 批寫入完成 (50,000 筆)
✅ 第 9 批寫入完成 (50,000 筆)
✅ 第 10 批寫入完成 (50,000 筆)
📊 資料表總筆數: 500,000 筆


In [13]:
with rdb_connection(datasource) as conn:
    with conn.begin():
        conn.execute(text("DROP TABLE  " + table_name ))
    print("✅ 刪除 " + table_name + " 成功")


✅ 刪除 tmp_massive_insert 成功
